# Separação dos dados em treino, validação e teste

## Estratégia

O objetivo é prever as kills da próxima partida usando o histórico do jogador. Por isso a divisão é feita **por data**, e não por sorteio: o treino usa as partidas mais antigas, a validação as seguintes e o teste as mais recentes. Assim, nenhuma partida futura é usada para treinar um modelo avaliado em partidas anteriores.

## Proporções

| Conjunto | Proporção | Uso |
|---|---|---|
| Treino | 70% | O modelo aprende |
| Validação | 15% | Comparar e ajustar modelos |
| Teste | 15% | Avaliação final, usada só depois das decisões |

Justificativa: com mais de 170 mil linhas, 15% (cerca de 26 mil) já é suficiente para avaliar com estabilidade, e 70% dá bastante dado para o modelo aprender.

## Ordem das etapas

1. Criar as informações do histórico (`criar_features_historico`). Elas usam só partidas anteriores, então calculá-las antes da divisão não vaza informação.
2. Manter só linhas com pelo menos 5 partidas anteriores (`filtrar_minimo_partidas`).
3. Dividir por data (`dividir_por_tempo`).



In [1]:
import sys
sys.path.append("..")

import pandas as pd

from src.preprocessing.feature_historico import criar_features_historico, filtrar_minimo_partidas
from src.preprocessing.separacao_dados import (
    dividir_por_tempo,
    resumo_separacao,
    verificar_ordem_temporal,
)

dados = pd.read_csv("../data/raw/tb_lobby_stats_player.csv")
df = filtrar_minimo_partidas(criar_features_historico(dados))

treino, validacao, teste = dividir_por_tempo(df)
verificar_ordem_temporal(treino, validacao, teste)

print("Linhas após o filtro de 5 partidas:", len(df))
display(resumo_separacao(treino, validacao, teste))


Linhas após o filtro de 5 partidas: 172863


,Linhas,Porcentagem (%),Primeira data,Última data,Jogadores
Treino,121004,70.0,2021-09-14 19:07:41,2022-01-08 23:56:35,1608
Validação,25929,15.0,2022-01-08 23:56:52,2022-01-26 03:47:23,1550
Teste,25930,15.0,2022-01-26 03:47:25,2022-02-11 13:27:48,1672


In [2]:
print("Soma dos conjuntos:", len(treino) + len(validacao) + len(teste))
print("Jogadores do teste que não aparecem no treino:",
      len(set(teste['idPlayer']) - set(treino['idPlayer'])), "de", teste['idPlayer'].nunique())

print("Média de kills (treino, validação, teste):",
      round(treino['qtKill'].mean(), 2), round(validacao['qtKill'].mean(), 2), round(teste['qtKill'].mean(), 2))


Soma dos conjuntos: 172863
Jogadores do teste que não aparecem no treino: 352 de 1672
Média de kills (treino, validação, teste): 19.2 19.12 19.1


## Resultados

- Os três conjuntos somam 172.863 linhas: treino 121.004 (70,0%), validação 25.929 (15,0%) e teste 25.930 (15,0%).
- Os períodos não se sobrepõem: o treino vai de 14/09/2021 a 08/01/2022, a validação de 08/01/2022 a 26/01/2022 e o teste de 26/01/2022 a 11/02/2022.
- A média de kills é parecida nos três conjuntos (19,20, 19,12 e 19,10), então a divisão não criou um conjunto muito diferente dos outros.
- 352 dos 1.672 jogadores do teste (21%) não têm nenhuma linha no treino. Isso não é problema, porque o `idPlayer` não é usado como entrada: o modelo olha o histórico do jogador, e não quem ele é.
